In [1]:
import time
import random
import numpy as np

# --- Блок генерации данных (Численное интегрирование нелинейного осциллятора) ---
dt = 2 * np.pi / 1000
k = 15
L = k / 100
omega = 1000 / k
N = 500

x = np.zeros(N)
x[1] = (-1) ** k * dt

# Конечно-разностная схема второго порядка (модифицированный метод Верле)
for i in range(2, N):
    # Моделируем траекторию нелинейной динамической системы с кубическим демпфированием
    x[i] = (x[i-1] * (2 + dt * L * (1 - x[i-1] ** 2)) -
            x[i-2] * (1 + dt ** 2 + dt * L * (1 - x[i-2] ** 2)) +
            dt ** 2 * np.sin(omega * i * dt))


# --- Блок оптимизации (Кастомные движки градиентного спуска на чистом NumPy) ---

def compute_full_gradient(x_true: np.ndarray, a: float, b: float, om1: float, om2: float, dt: float) -> tuple:
    """
    Вычисление точного аналитического градиента (Full-Batch) по всему массиву данных.
    Сложность: O(N)
    """
    t = np.arange(len(x_true)) * dt
    # Базис тригонометрического многочлена
    harmonics = np.sin(om1 * t) + np.cos(om1 * t) + np.sin(om2 * t) + np.cos(om2 * t)
    pred = a + b * harmonics
    error = pred - x_true

    # Частные производные функции потерь (MSE) по оптимизируемым параметрам
    grad_a0 = np.sum(error)
    grad_a1 = np.sum(error * np.cos(om1 * t))
    grad_a2 = np.sum(error * np.cos(om2 * t))
    grad_b1 = np.sum(error * np.sin(om1 * t))
    grad_b2 = np.sum(error * np.sin(om2 * t))

    # Производные по частотам (невыпуклый ландшафт оптимизации)
    grad_om1 = np.sum(error * (-b * np.sin(om1 * t) * t))
    grad_om2 = np.sum(error * (-b * np.sin(om2 * t) * t))

    return grad_a0, grad_a1, grad_a2, grad_b1, grad_b2, grad_om1, grad_om2


def compute_stochastic_gradient(x_true: np.ndarray, a: float, b: float, om1: float, om2: float, idx: int, dt: float) -> tuple:
    """
    Вычисление стохастического градиента строго в одной случайной точке временного ряда.
    Сложность: O(1)
    """
    t = idx * dt
    harmonics = np.sin(om1 * t) + np.cos(om1 * t) + np.sin(om2 * t) + np.cos(om2 * t)
    pred = a + b * harmonics
    error = pred - x_true[idx]

    # Локальные производные для шага по случайному сэмплу
    grad_a0 = error
    grad_a1 = error * np.cos(om1 * t)
    grad_a2 = error * np.cos(om2 * t)
    grad_om1 = error * (-b * np.sin(om1 * t) * t)
    grad_om2 = error * (-b * np.sin(om2 * t) * t)

    return grad_a0, grad_a1, grad_a2, grad_om1, grad_om2


def gradient_descent(x_true: np.ndarray, learning_rate: float, num_iterations: int, dt: float) -> tuple:
    """Классический Full-Batch Градиентный спуск."""
    a, b = np.random.rand(), np.random.rand()
    om1, om2 = np.random.rand() * omega, np.random.rand() * omega

    for _ in range(num_iterations):
        grad_a0, grad_a1, grad_a2, _, _, grad_om1, grad_om2 = compute_full_gradient(x_true, a, b, om1, om2, dt)

        # Обновление весов с учетом накопленного градиента по всему пакету
        a -= learning_rate * grad_a0
        b -= learning_rate * (grad_a1 + grad_a2)
        om1 -= learning_rate * grad_om1
        om2 -= learning_rate * grad_om2

    return a, b, om1, om2


def minimize_stochastic(x_true: np.ndarray, learning_rate: float, num_iterations: int, dt: float) -> tuple:
    """Честный Стохастический Градиентный Спуск (SGD)."""
    a, b = np.random.rand(), np.random.rand()
    om1, om2 = np.random.rand() * omega, np.random.rand() * omega
    data_size = len(x_true)

    for _ in range(num_iterations):
        # Выбираем один случайный индекс на итерацию
        idx = np.random.randint(data_size)

        # Считаем градиент только для этой точки (O(1))
        grad_a0, grad_a1, grad_a2, grad_om1, grad_om2 = compute_stochastic_gradient(x_true, a, b, om1, om2, idx, dt)

        a -= learning_rate * grad_a0
        b -= learning_rate * (grad_a1 + grad_a2)
        om1 -= learning_rate * grad_om1
        om2 -= learning_rate * grad_om2

    return a, b, om1, om2


# --- Тесты производительности и профилирование ---
if __name__ == "__main__":
    print(f"[INFO] Длина сигнального вектора N = {N}. Запуск бенчмарка...")

    iterations = 2000
    lr = 0.001

    # Профилирование Full-Batch GD
    start_gd = time.time()
    res_a_gd, res_b_gd, res_om1_gd, res_om2_gd = gradient_descent(x, lr, iterations, dt)
    gd_time = time.time() - start_gd

    # Профилирование честного SGD
    start_sgd = time.time()
    res_a_sgd, res_b_sgd, res_om1_sgd, res_om2_sgd = minimize_stochastic(x, lr, iterations, dt)
    sgd_time = time.time() - start_sgd

    print("\n" + "="*50)
    print(" СРАВНИТЕЛЬНЫЙ АНАЛИЗ ВРЕМЕНИ ВЫЧИСЛЕНИЙ")
    print("="*50)
    print(f"Пакетный градиентный спуск (GD):  {gd_time:.6f} сек.")
    print(f"Стохастический спуск (Честный SGD): {sgd_time:.6f} сек.")
    print("-"*50)
    print(f"Ускорение SGD относительно GD:     {gd_time / sgd_time:.2f}x")
    print("="*50 + "\n")

    print("[DEBUG] Результаты GD:  a={:.4f}, b={:.4f}, om1={:.4f}, om2={:.4f}".format(res_a_gd, res_b_gd, res_om1_gd, res_om2_gd))
    print("[DEBUG] Результаты SGD: a={:.4f}, b={:.4f}, om1={:.4f}, om2={:.4f}".format(res_a_sgd, res_b_sgd, res_om1_sgd, res_om2_sgd))

[INFO] Длина сигнального вектора N = 500. Запуск бенчмарка...

 СРАВНИТЕЛЬНЫЙ АНАЛИЗ ВРЕМЕНИ ВЫЧИСЛЕНИЙ
Пакетный градиентный спуск (GD):  0.578332 сек.
Стохастический спуск (Честный SGD): 0.061564 сек.
--------------------------------------------------
Ускорение SGD относительно GD:     9.39x

[DEBUG] Результаты GD:  a=-0.6490, b=-0.0189, om1=13.5582, om2=23.5825
[DEBUG] Результаты SGD: a=-0.5360, b=0.1025, om1=54.4263, om2=40.5357
